# M01 — XGBoost: Gradient Boosted Trees

## Why XGBoost wins competitions and interviews

XGBoost is gradient boosting done right. It adds three key improvements over vanilla gradient boosting:
1. **Regularization** (L1/L2) — prevents overfitting that sklearn's GradientBoostingClassifier ignores
2. **Parallel tree construction** — builds each tree level in parallel, not node by node
3. **Sparsity awareness** — handles missing values natively with a learned default direction

## The math you must know

At each boosting round $m$, we fit a tree $f_m$ to the **pseudo-residuals** of the current ensemble:

$$F_m(x) = F_{m-1}(x) + \eta \cdot f_m(x)$$

XGBoost uses a **second-order Taylor expansion** of the loss to compute the optimal leaf weights:

$$\text{Leaf weight}^* = -\frac{\sum_i g_i}{\sum_i h_i + \lambda}$$

where $g_i = \partial L / \partial \hat{y}_i$ (first derivative, gradient) and $h_i = \partial^2 L / \partial \hat{y}_i^2$ (second derivative, Hessian), and $\lambda$ is the L2 regularization term.

**The gain formula for a split:**
$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

A split is only made if Gain > 0. $\gamma$ (min_split_loss) is a regularization parameter that prunes splits with low gain.

**Reference:** [XGBoost docs](https://xgboost.readthedocs.io/en/stable/)


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Credit (classification)
credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')
credit['target'] = (credit['class'] == 'good').astype(int)

# Housing (regression)
housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
X_reg = housing.drop('medhousval', axis=1)
y_reg = housing['medhousval']

print(f"Credit: {credit.shape} | Housing: {housing.shape}")
print(f"XGBoost version: {xgb.__version__}")

---
## Exercise 1 — XGBoost Classifier: Baseline

**Task:** Train your first XGBoost classifier and understand the key parameters.

1. Prepare the credit dataset: encode all categoricals with `OrdinalEncoder`, impute nulls.
2. Split 80/20, stratified.
3. Train `xgb.XGBClassifier` with: `n_estimators=100`, `max_depth=4`, `learning_rate=0.1`, `eval_metric='auc'`, `random_state=42`.
4. Enable **early stopping**: pass `eval_set=[(X_test, y_test)]` and `early_stopping_rounds=10` — training stops when the metric stops improving.
5. Report: `best_iteration`, `best_score`, test AUC, test F1.

**Key parameter to understand:** What does `learning_rate` do mathematically? (See the $\eta$ in the formula above.)

In [ ]:
# Prepare features
cat_cols = credit.select_dtypes(include='object').columns.drop('class').tolist()
num_cols = credit.select_dtypes(include='number').columns.drop('target').tolist()

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
])

X = credit[num_cols + cat_cols]
y = credit['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# YOUR CODE HERE
clf = None
best_iteration = None
best_score = None
test_auc = None
test_f1 = None

In [ ]:
# --- ASSERTIONS ---
assert clf is not None
assert isinstance(clf, xgb.XGBClassifier)
assert best_iteration is not None and best_iteration <= 100
assert test_auc > 0.65, f"AUC must be > 0.65, got {test_auc:.4f}"
assert test_f1 > 0.60
print(f"✓ Exercise 1 passed")
print(f"Best iteration: {best_iteration} | Best CV AUC: {best_score:.4f}")
print(f"Test AUC: {test_auc:.4f} | Test F1: {test_f1:.4f}")

**learning_rate explanation:** *(What does $\eta$ control mathematically? What happens if it's too high vs too low?)*

---
## Exercise 2 — XGBoost Regressor: Housing Prices

**Task:** Train an XGBoost regressor and understand objective functions.

1. Train `xgb.XGBRegressor` on housing data with `objective='reg:squarederror'`.
2. Implement custom RMSE and RMSLE (root mean squared log error) evaluation using the `custom_metric` parameter.
   - RMSLE = `sqrt(mean((log(y_pred+1) - log(y_true+1))^2))`
3. Compare models trained with `objective='reg:squarederror'` vs `objective='reg:absoluteerror'`.
4. Report: RMSE, MAE, R² for both on the test set.

**Key concept:** Why would you choose `absoluteerror` over `squarederror`? Write the answer below.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# YOUR CODE HERE
reg_mse = None
reg_mae = None
comparison_df = None  # DataFrame comparing both objectives

In [ ]:
# --- ASSERTIONS ---
assert comparison_df is not None
assert len(comparison_df) == 2
assert 'rmse' in comparison_df.columns or 'RMSE' in comparison_df.columns
r2_col = [c for c in comparison_df.columns if 'r2' in c.lower() or 'r_sq' in c.lower()][0]
assert (comparison_df[r2_col] > 0.7).all(), "Both models must achieve R² > 0.7"
print("✓ Exercise 2 passed")
print(comparison_df.to_string(index=False))

**absoluteerror vs squarederror:** *(When would absolute error loss be preferred? What type of outlier behavior does each incentivize?)*

---
## Exercise 3 — Hyperparameter Tuning: The Key Parameters

**The XGBoost hyperparameters that matter most** (in order of importance):

| Parameter | Controls | Typical range |
|---|---|---|
| `n_estimators` | Number of trees | 100–2000 |
| `learning_rate` | Shrinkage $\eta$ | 0.01–0.3 |
| `max_depth` | Tree complexity | 3–8 |
| `subsample` | Row sampling per tree | 0.5–1.0 |
| `colsample_bytree` | Feature sampling per tree | 0.5–1.0 |
| `reg_alpha` | L1 regularization | 0–10 |
| `reg_lambda` | L2 regularization | 0–10 |
| `min_child_weight` | Min hessian sum in leaf | 1–10 |

**Task:** Run a `RandomizedSearchCV` over this parameter space. But first — implement the **learning rate / n_estimators relationship** correctly:
- Lower learning_rate → need more n_estimators
- Use a grid where these are inversely related: `[(lr=0.3, n=100), (lr=0.1, n=300), (lr=0.05, n=600)]`

Return `tuning_results`: DataFrame of top 10 parameter combinations with their CV AUC scores.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# YOUR CODE HERE
tuning_results = None
best_xgb = None

In [ ]:
# --- ASSERTIONS ---
assert tuning_results is not None
assert len(tuning_results) == 10
assert 'mean_test_score' in tuning_results.columns
assert tuning_results['mean_test_score'].is_monotonic_decreasing
assert tuning_results['mean_test_score'].iloc[0] > 0.70
print(f"✓ Exercise 3 passed — Best CV AUC: {tuning_results['mean_test_score'].iloc[0]:.4f}")
print(tuning_results[['mean_test_score','std_test_score']].head())

---
## Exercise 4 — Native DMatrix API

**Concept:** `xgb.DMatrix` is XGBoost's native data format — faster than numpy arrays and supports missing values natively.

1. Convert train and test sets to `xgb.DMatrix`, including feature names.
2. Train using `xgb.train()` (the low-level API) with a watchlist.
3. Implement a custom evaluation metric: **Brier score** = `mean((y_pred - y_true)^2)`.
4. Extract training history from `evals_result` — show AUC and Brier score per round.
5. Return `training_history`: DataFrame with columns `round`, `train_auc`, `test_auc`, `train_brier`, `test_brier`.

In [ ]:
def brier_score(y_pred: np.ndarray, dtrain: xgb.DMatrix):
    """
    Custom Brier score metric for XGBoost.
    Returns: (metric_name, metric_value) — lower is better.
    """
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: DMatrix, xgb.train(), training history
training_history = None

In [ ]:
# --- ASSERTIONS ---
assert training_history is not None
for col in ['round','train_auc','test_auc']:
    assert col in training_history.columns, f"Missing: {col}"
assert training_history['test_auc'].iloc[-1] > 0.65
# Brier score should decrease over training
if 'train_brier' in training_history.columns:
    first_brier = training_history['train_brier'].iloc[0]
    last_brier = training_history['train_brier'].iloc[-1]
    assert last_brier <= first_brier, "Brier score must improve (decrease) over training"
print("✓ Exercise 4 passed")
print(training_history.tail())

---
## Exercise 5 — Feature Importance: 4 Methods

**XGBoost provides 4 types of feature importance — and they often disagree:**

| Type | What it measures | When to use |
|---|---|---|
| `weight` | # times feature used in splits | Quick overview |
| `gain` | Average gain of splits using this feature | Most interpretable |
| `cover` | Average sample coverage in splits | Data volume proxy |
| `total_gain` | Total gain across all splits | Dominated by frequent features |

1. Extract all 4 importance types from the trained model.
2. Build `importance_df`: rows = features, columns = weight, gain, cover, total_gain. Normalize each to sum to 1.
3. Compute `rank_correlation`: Spearman correlation between each pair of importance measures.
4. Identify features where `weight` rank and `gain` rank disagree most — these features are used often but contribute little per split.

In [ ]:
from scipy.stats import spearmanr

# YOUR CODE HERE
importance_df = None
rank_correlation = None
disagreement_features = None  # features where weight rank != gain rank by > N positions

In [ ]:
# --- ASSERTIONS ---
assert importance_df is not None
for col in ['weight','gain','cover','total_gain']:
    assert col in importance_df.columns
    assert abs(importance_df[col].sum() - 1.0) < 0.01, f"{col} must sum to 1"
assert rank_correlation is not None
assert isinstance(disagreement_features, list)
print("✓ Exercise 5 passed")
print(importance_df.sort_values('gain', ascending=False).head(8))
print(f"\nDisagreement features: {disagreement_features[:5]}")

---
## Exercise 6 — Handling Imbalanced Classes

**Concept:** XGBoost has a native parameter for class imbalance: `scale_pos_weight` = (negative samples) / (positive samples). This adjusts the gradient computation to weight the minority class more heavily.

1. Compute the correct `scale_pos_weight` from the training set.
2. Train two models: one with `scale_pos_weight=1` (default), one with the computed weight.
3. Compare recall and precision for the minority class across both models.
4. Also compare with `class_weight='balanced'` in sklearn's LogisticRegression as a reference.
5. Return `imbalance_comparison`: DataFrame with model, precision, recall, f1, roc_auc.

In [ ]:
# YOUR CODE HERE
imbalance_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert imbalance_comparison is not None
assert len(imbalance_comparison) == 3
for col in ['precision','recall','f1','roc_auc']:
    assert col in imbalance_comparison.columns
# Weighted model should have higher recall for minority class
models = imbalance_comparison['model'].tolist() if 'model' in imbalance_comparison.columns else []
print("✓ Exercise 6 passed")
print(imbalance_comparison.to_string(index=False))

---
## Exercise 7 — XGBoost in sklearn Pipeline

**Task:** Integrate XGBoost into a full sklearn Pipeline that handles preprocessing + model + cross-validation correctly.

1. Build `full_pipeline`: `ColumnTransformer` (median impute + ordinal encode) → `XGBClassifier`
2. Run 5-fold stratified CV with `cross_val_score` — scoring='roc_auc'
3. Use `GridSearchCV` to tune `max_depth` (3, 5, 7) and `n_estimators` (100, 200) inside the pipeline. Note: use `xgbclassifier__max_depth` syntax.
4. Verify the pipeline is cloneable: `sklearn.base.clone(full_pipeline)` must work.
5. Report final test AUC.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

# YOUR CODE HERE
full_pipeline = None
cv_scores = None
grid_result = None

In [ ]:
# --- ASSERTIONS ---
assert isinstance(full_pipeline, Pipeline)
assert cv_scores is not None and len(cv_scores) == 5
assert cv_scores.mean() > 0.65
cloned = clone(full_pipeline)
assert not hasattr(cloned.steps[-1][1], 'n_estimators_') or True  # unfitted
if grid_result is not None:
    assert grid_result.best_score_ > 0.65
print(f"✓ Exercise 7 passed — CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## Exercise 8 — SHAP Values with XGBoost

**Concept:** SHAP (SHapley Additive exPlanations) values explain individual predictions — not just global importance.

XGBoost computes SHAP values natively (fast, exact for trees).

1. Compute SHAP values using `model.get_booster().predict(dmatrix, pred_contribs=True)`.
2. Build `shap_df`: rows = test samples, columns = features + `bias`. Shape: `(n_test, n_features + 1)`.
3. Verify: for each sample, `sum(shap_values) + bias ≈ model output (log-odds)`.
4. Compute `mean_abs_shap`: mean absolute SHAP value per feature — sorted descending. This is the SHAP global importance.
5. Find the 3 test samples with the highest `prediction uncertainty` — defined as the sample where the top feature's SHAP value dominates least (lowest `max(|shap|) / sum(|shap|)` ratio).

In [ ]:
# YOUR CODE HERE
shap_df = None
mean_abs_shap = None
uncertain_samples = None

In [ ]:
# --- ASSERTIONS ---
assert shap_df is not None
assert 'bias' in shap_df.columns or shap_df.shape[1] == len(num_cols) + len(cat_cols) + 1
# SHAP values sum check (verify additivity)
dtest = xgb.DMatrix(X_test_enc)
raw_preds = clf.get_booster().predict(dtest, output_margin=True)
shap_sum = shap_df.sum(axis=1).values
assert np.allclose(shap_sum, raw_preds, atol=1e-3), "SHAP values must sum to model output"

assert mean_abs_shap is not None
assert mean_abs_shap.is_monotonic_decreasing
assert len(uncertain_samples) == 3
print("✓ Exercise 8 passed")
print("Top 5 features by mean |SHAP|:")
print(mean_abs_shap.head())

---
## Exercise 9 — Monotone Constraints

**Business context:** In credit risk, the model must be monotone: higher income should always lead to lower predicted default probability, regardless of other features. XGBoost can enforce this with `monotone_constraints`.

1. Train a baseline XGBoost without constraints.
2. Train with `monotone_constraints`: force `credit_amount` (index in feature matrix) to be positively monotone with target (more credit = less likely to be 'good'? — set to -1), and `age` positively monotone (+1).
3. Verify the constraint holds: vary `credit_amount` while holding all other features fixed — the predicted probability must be monotonically decreasing.
4. Compare test AUC between constrained and unconstrained models.

In [ ]:
# YOUR CODE HERE
model_unconstrained = None
model_constrained = None
monotone_check_passed = None  # bool

In [ ]:
# --- ASSERTIONS ---
assert model_constrained is not None
assert isinstance(monotone_check_passed, bool)
assert monotone_check_passed, "Monotonicity constraint must hold for credit_amount"
auc_unconstrained = roc_auc_score(y_test, model_unconstrained.predict_proba(X_test_enc)[:, 1])
auc_constrained = roc_auc_score(y_test, model_constrained.predict_proba(X_test_enc)[:, 1])
print(f"✓ Exercise 9 passed")
print(f"Unconstrained AUC: {auc_unconstrained:.4f} | Constrained AUC: {auc_constrained:.4f}")
print(f"AUC cost of constraint: {auc_unconstrained - auc_constrained:.4f}")

---
## Exercise 10 — Capstone: Full XGBoost Production Workflow

**Spec:** Build a production-ready XGBoost model with the complete workflow a DS would follow.

1. **Data prep**: full preprocessing pipeline (imputation + encoding + feature engineering: add interaction `credit_amount * duration`, log transform skewed features)
2. **Baseline**: logistic regression benchmark
3. **XGBoost tuning**: RandomizedSearchCV (n_iter=20) over full parameter space
4. **Calibration**: calibrate probabilities with `CalibratedClassifierCV` and verify with Brier score
5. **Threshold optimization**: find threshold maximizing F1 on validation set
6. **Final evaluation**: test set metrics at optimal threshold (precision, recall, F1, AUC, Brier)
7. **Save model**: serialize full pipeline to `/tmp/credit_model.joblib`
8. **Load and verify**: reload and assert predictions match

Return `final_report`: dict with all metrics.

In [ ]:
import joblib
from sklearn.calibration import CalibratedClassifierCV

# YOUR CODE HERE
final_report = None

In [ ]:
# --- ASSERTIONS ---
import os
assert final_report is not None
required_keys = ['test_auc','test_f1','test_precision','test_recall','brier_score','optimal_threshold']
for k in required_keys:
    assert k in final_report, f"Missing: {k}"
assert final_report['test_auc'] > 0.70
assert os.path.exists('/tmp/credit_model.joblib')

# Reload and verify
loaded_model = joblib.load('/tmp/credit_model.joblib')
X_sample = X_test[:5]
orig_preds = loaded_model.predict_proba(X_sample)
assert orig_preds.shape == (5, 2)

print("✓ Exercise 10 passed — Full XGBoost workflow complete")
for k, v in final_report.items():
    print(f"  {k}: {v:.4f}")